# Download for multiple Tickers of info

In [1]:
import yfinance as yf 
import pandas as pd 

tic_apple = yf.Ticker("AAPL")
tic_info = tic_apple.info
tic_financial  = tic_apple.financials
tic_balance = tic_apple.balance_sheet
tic_apple_cash = tic_apple.cash_flow

tic_financial.loc["date"] = tic_financial.columns.tolist()



ModuleNotFoundError: No module named 'yfinance'

In [2]:
from sqlalchemy.orm import DeclarativeBase,Mapped,mapped_column
from sqlalchemy import String,DateTime,func
from datetime import datetime
from typing import Dict
from sqlalchemy.dialects.postgresql import JSONB
class Base(DeclarativeBase):
    pass
class StockDataInfo(Base):
    __tablename__ = "info"
    __table_args__ = {"schema":"market_data"}

    symbol:Mapped[str] = mapped_column(String(10), primary_key=True)
    jsondump: Mapped[Dict[str,str]] =mapped_column(JSONB)
    created_at:Mapped[datetime] = mapped_column(DateTime(timezone=True), server_default=func.now())

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
database_url = os.getenv("DATABASE_URL")
# get symbols
symbols = ["AAPL", "A", "TSLA"]
#pd.read_csv("sp500_companies.csv")["Symbol"].to_list()

from sqlalchemy import create_engine
from sqlalchemy.orm import  sessionmaker
engine = create_engine(database_url, echo=True, pool_pre_ping=True)

# this for creating the db-Entry per Ticker to stagingDB
def create_stockDataInfo(symbol):
    
    ticker = yf.Ticker(symbol)
    info = ticker.info 
    stock_db = StockDataInfo(symbol=symbol, jsondump=info)
    return stock_db
# this only sets up the staging DB
Base.metadata.create_all(bind = engine)

Session = sessionmaker(engine)

with Session.begin() as session:
    for symbol in symbols:
        try:
            stock_add = create_stockDataInfo(symbol)
            session.merge(stock_add)
            
        except Exception as e:
            print(f"Following {symbol} failed: {e}")




2026-03-25 09:20:36,826 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-25 09:20:36,827 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-25 09:20:36,829 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-25 09:20:36,830 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-25 09:20:36,833 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-25 09:20:36,833 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-25 09:20:36,835 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-25 09:20:36,838 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_namespace.nspname = %(nspname_1)s
2026-03-25 09:20:36,839 INFO sqlalchemy.engine.Engine [g

# Creating the Table for Company Data

In [4]:
from pathlib import Path
from sqlalchemy import text 
path = Path.cwd().parent / "sql" /"create_tables_info_balance_financials.sql"
print(path)
sql_stmt_create_table = path.read_text(encoding="utf-8")

with Session.begin() as session:
    session.execute(text(sql_stmt_create_table))

c:\Users\ASUS\projekte\react_1\e-commerce\api\api_yahoo\sql\create_tables_info_balance_financials.sql
2026-03-25 09:20:37,905 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-25 09:20:37,906 INFO sqlalchemy.engine.Engine CREATE TABLE IF NOT EXISTS market_data.company_profile (
symbol VARCHAR(20) PRIMARY KEY,
short_name TEXT,
long_name TEXT,
quote_type TEXT,
exchange TEXT,
currency TEXT,
website TEXT,
sector TEXT,
industry TEXT,
country TEXT,
city TEXT,
state TEXT,
full_time_employees BIGINT,
long_business_summary TEXT,
market_cap NUMERIC(20,0),
enterprise_value NUMERIC(20,0),
total_revenue NUMERIC(20,0),
ebitda NUMERIC(20,0),
revenue_growth NUMERIC(12,6),
gross_margins NUMERIC(12,6),
operating_margins NUMERIC(12,6),
profit_margins NUMERIC(12,6),
free_cashflow NUMERIC(20,0),
trailing_pe NUMERIC(14,6),
forward_pe NUMERIC(14,6),
peg_ratio NUMERIC(14,6),
price_to_book NUMERIC(14,6),
trailing_eps NUMERIC(14,6),
forward_eps NUMERIC(14,6),
return_on_equity NUMERIC(14,6),
return_on_asset

In [5]:
path = Path.cwd().parent /"sql" /"update_company_profile.sql"
sql_stmt_staging_to_productive= path.read_text(encoding="utf-8")
with Session.begin() as session:
    session.execute(text(sql_stmt_staging_to_productive))


2026-03-25 09:20:37,921 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-25 09:20:37,921 INFO sqlalchemy.engine.Engine WITH latest_info AS (
SELECT x.*
FROM (
SELECT
i.*,
ROW_NUMBER() OVER (PARTITION BY i.symbol ORDER BY i.created_at DESC NULLS LAST) AS rn
FROM market_data.info i
) x
WHERE x.rn = 1
)
INSERT INTO market_data.company_profile (
symbol, short_name, long_name, quote_type, exchange, currency, website,
sector, industry, country, city, state, full_time_employees, long_business_summary,
market_cap, enterprise_value, total_revenue, ebitda, revenue_growth, gross_margins,
operating_margins, profit_margins, free_cashflow, trailing_pe, forward_pe, peg_ratio,
price_to_book, trailing_eps, forward_eps, return_on_equity, return_on_assets, debt_to_equity,
current_ratio, quick_ratio, dividend_rate, dividend_yield, payout_ratio, ex_dividend_date,
beta, shares_outstanding, average_volume, fifty_two_week_high, fifty_two_week_low,
fifty_day_average, two_hundred_day_average, source_creat

## Adding the financials, balance sheet and cash flows 


In [6]:
class StockDataFinancialsStaging(Base):
    __tablename__ = "financials_staging"
    __table_args__ = {"schema": "market_data"}

    symbol : Mapped[str] = mapped_column(String(10), primary_key = True)
    jsondump:Mapped[Dict[str,str]] = mapped_column(JSONB)
    created_at:Mapped[datetime] = mapped_column (DateTime(timezone=True), server_default = func.now())




In [7]:
database_url = os.getenv("DATABASE_URL")
engine = create_engine(database_url,echo = True, pool_pre_ping =True )
session = sessionmaker(bind = engine)

symbols = ["AAPL", "A", "TSLA"]

def create_financials(symbol):
    ticker = yf.Ticker(symbol)
    fin = ticker.financials

    fin = fin.rename_axis("metric").stack().reset_index(name="value")
    fin["year"] = pd.to_datetime(fin["level_1"], errors = "coerce").dt.year
    fin.drop(columns=["level_1"], inplace=True)
   
    # making NaN to None for valid JSON
    fin = fin.astype(object).where(pd.notna(fin), None)
    payload = fin.to_dict(orient = "records")

    new_entry = StockDataFinancialsStaging(symbol= symbol, jsondump = payload)
    return new_entry

Base.metadata.create_all(bind = engine)

with session.begin() as sess:
    for symbol in symbols:
        new_entry = create_financials(symbol)
        try:
            sess.merge(new_entry)
        except Exception as e:
            print(f"Following error occured {e} on symbol {symbol}")

2026-03-25 09:20:38,221 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-25 09:20:38,221 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-25 09:20:38,223 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-25 09:20:38,223 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-25 09:20:38,225 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-25 09:20:38,226 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-25 09:20:38,228 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-25 09:20:38,229 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_namespace.nspname = %(nspname_1)s
2026-03-25 09:20:38,230 INFO sqlalchemy.engine.Engine [g